# Step 4: Batch Prediction & Inference
This notebook demonstrates how to load the trained models and run inference on an unseen dataset to generate poverty predictions and cluster assignments. The logic is self-contained.

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import joblib
import lightgbm as lgb

## 1. Defining Prediction Logic
We define a function to load the models and run predictions on engineered data.

In [8]:
def batch_predict(input_csv, output_csv, model_path):
    """Perform batch prediction on an unseen dataset."""
    print(f"Loading models from {model_path}...")
    classifier = lgb.Booster(model_file=os.path.join(model_path, 'lgbm_model.txt'))
    kmeans = joblib.load(os.path.join(model_path, 'kmeans_model.joblib'))
    scaler = joblib.load(os.path.join(model_path, 'scaler.joblib'))
    
    df = pd.read_csv(input_csv)
    X = df.drop(columns=['Id', 'idhogar', 'Target', 'Cluster'], errors='ignore')
    
    print("Generating predictions...")
    X_scaled = scaler.transform(X.fillna(0))
    clusters = kmeans.predict(X_scaled)
    
    y_prob = classifier.predict(X)
    y_pred = np.argmax(y_prob, axis=1) + 1
    
    results = pd.DataFrame({
        'Id': df['Id'],
        'Household_Id': df['idhogar'],
        'Poverty_Prediction': y_pred,
        'Cluster_Assignment': clusters
    })
    
    results.to_csv(output_csv, index=False)
    return results

## 2. Running Inference
We apply the models to the engineered test data.

In [9]:
MODEL_PATH = '../models/'
TEST_DATA_PATH = '../data/processed/test_engineered.csv'
OUTPUT_PATH = '../data/processed/final_predictions.csv'

if os.path.exists(TEST_DATA_PATH):
    results = batch_predict(TEST_DATA_PATH, OUTPUT_PATH, MODEL_PATH)
    display(results.head())
    print(f"\nPredictions saved to {OUTPUT_PATH}")
else:
    print("Test data not found. Please run the engineering notebook first.")

Loading models from ../models/...
Generating predictions...


,Id,Household_Id,Poverty_Prediction,Cluster_Assignment
0,ID_e5442cf6a,72958b30c,4,3
1,ID_a8db26a79,5b598fbc9,4,3
2,ID_a62966799,1e2fc704e,4,3
3,ID_3c5f4bd51,8ee7365a8,4,3
4,ID_472fa82da,ff69a6fc8,4,3



Predictions saved to ../data/processed/final_predictions.csv


## 3. Sanity Check: Distribution Overlap
Since we don't have the labels for the test set, we check if our predictions look "realistic" by comparing them to the training distribution.

In [10]:
train_df = pd.read_csv('../data/processed/train_engineered.csv')
train_dist = train_df['Target'].value_counts(normalize=True).sort_index()
pred_dist = results['Poverty_Prediction'].value_counts(normalize=True).sort_index()

dist_df = pd.DataFrame({
    'Label': train_dist.index.astype(str),
    'Train_Dist': train_dist.values,
    'Pred_Dist': pred_dist.values
})

fig = px.bar(dist_df, x='Label', y=['Train_Dist', 'Pred_Dist'], 
             barmode='group', title='Prediction Sanity Check: Training vs Test Distribution')
fig.show()

print("Interpretation: If the bars are roughly the same height, the model's logic is consistent across datasets.")

Interpretation: If the bars are roughly the same height, the model's logic is consistent across datasets.
